In [1]:
import pandas as pd
import lightgbm as lgb

train = pd.read_csv('../preprocessing/train.csv')
val   = pd.read_csv('../preprocessing/val.csv')

# X 컬럼 (date는 제외 — 분할용이었음)
X_cols = ['station_id', 'hour', 'dow', 'is_weekend', 'month', 'is_holiday',
          'temp', 'humid', 'wind', 'precip', 'pm10', 'pm25', 'ozone']

def prep(df):
    X = df[X_cols].copy()
    X['station_id'] = X['station_id'].astype('category')   # LightGBM에 범주형이라고 알려줌
    return X

X_train, X_val = prep(train), prep(val)
y_train_rent,   y_val_rent   = train['rent_cnt'],   val['rent_cnt']
y_train_return, y_val_return = train['return_cnt'], val['return_cnt']

print(f"train {X_train.shape} / val {X_val.shape}")

train (150024, 13) / val (14136, 13)


In [3]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
# LightGBM과 다른 점:
#  - objective 이름이 'count:poisson'
#  - 범주형은 enable_categorical=True 필요
#  - early stopping을 생성자에서 지정
model_rent_xgb = xgb.XGBRegressor(
    n_estimators=2000, learning_rate=0.05,
    objective='count:poisson',
    enable_categorical=True,
    early_stopping_rounds=50,
    random_state=42,
)

model_rent_xgb.fit(
    X_train, y_train_rent,
    eval_set=[(X_val, y_val_rent)],
    verbose=100,
)

mae_xgb = mean_absolute_error(y_val_rent, model_rent_xgb.predict(X_val))
print(f"XGBoost 대여 val MAE: {mae_xgb:.3f}  (LightGBM: 0.504)")

[0]	validation_0-poisson-nloglik:0.97013
[100]	validation_0-poisson-nloglik:0.66969
[200]	validation_0-poisson-nloglik:0.65915
[209]	validation_0-poisson-nloglik:0.65866
XGBoost 대여 val MAE: 0.510  (LightGBM: 0.504)
